# BD Plate Reader: export fp16 models (no training)

The int8 export squeezes the confidence scores into a few levels (the plate score tops out at 0.50 on every photo), which breaks the threshold rules and ties in NMS. This notebook exports **half precision (fp16)** models from the saved `best.pt` files instead. It trains nothing and keeps the int8 files untouched. The fp16 files are about twice as big (roughly 12 MB for both, still under the 15 MB target).

Before you start:
1. Runtime menu, Change runtime type, choose a GPU (T4 is fine).
2. Drive must already contain `MyDrive/bd-plate-reader/outputs/plate_detector/weights/best.pt` and `MyDrive/bd-plate-reader/outputs/char_detector/weights/best.pt`, and `MyDrive/bd-plate-reader/data` with the dataset archives.
3. Run the cells from top to bottom.


In [ ]:
# Settings.
REPO_URL = "https://github.com/Kanto065/bd-plate-reader.git"
BRANCH = "main"


In [ ]:
# Connect Drive and check the GPU.
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi -L
!ls -lh "/content/drive/MyDrive/bd-plate-reader/data"

In [ ]:
# Get the code and install requirements.
import os
if not os.path.isdir("/content/bd-plate-reader"):
    !git clone -b {BRANCH} {REPO_URL} /content/bd-plate-reader
else:
    !git -C /content/bd-plate-reader pull
%cd /content/bd-plate-reader/training
!pip install -q -r requirements.txt

## Prepare the data
Unpacks the archives and rebuilds the datasets. Export needs the validation images for int8 calibration, and evaluation needs the test sets. This does not retrain anything and takes a few minutes.

In [ ]:
!python organize_data.py
!python merge_datasets.py

## Export to fp16 TFLite
This writes `plate_detector_fp16.tflite` and `char_detector_fp16.tflite` to Drive `bd-plate-reader/outputs/models`. The first export installs TensorFlow packages and can take a few minutes.

In [ ]:
import os
os.environ["BD_EXPORT_DTYPE"] = "fp16"
!python export.py


In [ ]:
# What was saved to Drive.
!ls -lh /content/drive/MyDrive/bd-plate-reader/outputs/models
print()
print("NEXT STEPS")
print("1. Open this Drive folder: My Drive / bd-plate-reader / outputs / models")
print("2. Download plate_detector_fp16.tflite and char_detector_fp16.tflite")
print("3. Give them to Claude and say which folder they are in.")
